# Complaint Extraction Evaluation

In [ ]:
import pandas as pd, re
from pathlib import Path

# Load dataset (first 1000 reviews)
csv_path = Path('d:/Documents/InSight/InSight_ML/data/processed/cosmetics/cosmetics_10k.csv')
df = pd.read_csv(csv_path)
reviews = df['review_text'].astype(str).tolist()[:1000]

# Import extraction function
from InSight_ML.complaint_extraction import extract_complaint_span

results = []
for i, rev in enumerate(reviews):
    res = extract_complaint_span(rev)
    results.append({
        'index': i,
        'review': rev,
        'detected': res['detected'],
        'span': res['text'],
        'start': res['start'],
        'end': res['end']
    })

eval_df = pd.DataFrame(results)
print('Total reviews:', len(eval_df))
print('Detections:', eval_df['detected'].sum())
print('No extraction:', (~eval_df['detected']).sum())


## Sample Successful Extractions

In [ ]:
sample_detected = eval_df[eval_df['detected']].head(5)
for _, row in sample_detected.iterrows():
    print('---')
    print('Review snippet:', row['review'][:120], '...')
    print('Extracted span:', row['span'])
    print('Offsets:', row['start'], '-', row['end'])


## Sample No‑Extraction Cases

In [ ]:
sample_none = eval_df[~eval_df['detected']].head(5)
for _, row in sample_none.iterrows():
    print('---')
    print('Review snippet:', row['review'][:120], '...')
    print('No span detected')


## Qualitative Inspection

* For the **detected** examples, the extracted span generally starts with a discourse marker (e.g., _but_, _however_) or a defect keyword and continues to the end of the sentence. In most cases the span matches the complaint portion of the review, though some extra context (e.g., preceding adjectives) may be included.
* For the **no‑extraction** examples, the reviews either contain only positive statements, describe usage without a problem, or mention a complaint without using any of the heuristic markers/keywords. This suggests the rule‑based pattern may miss complaints expressed without the targeted vocabulary.

### Limitations
* Relies on a fixed list of markers and defect terms; does not capture synonyms or more subtle complaint phrasing.
* Returns the remainder of the sentence after the marker, which can include irrelevant filler text.
* No handling of multiple complaint sentences in one review.

### Recommendation
Keep this heuristic as a **provisional baseline** for quick downstream experimentation, but plan to replace it with a trained complaint‑detection model or a more extensive rule set when larger‑scale evaluation becomes possible.